In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))  # add repo root
from rsr import rsr
import torch
import json
import pandas as pd

from ndtools import fun_binary_graph as fbg # ndtools available at github.com/jieunbyun/network-datasets
from ndtools.graphs import build_graph
from pathlib import Path
import networkx as nx   

# Load data

In [2]:
DATASET = Path("data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs_eq.json").read_text(encoding="utf-8"))

# build base graph
G_base: nx.Graph = build_graph(nodes, edges, probs_dict)


In [3]:
row_names = list(edges.keys())
n_state = 2 # binary states

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

In [4]:
dests = ['n22', 'n66']
sys_upper_st = 2

def s_fun(comps_st):
    conn_pop_ratio, sys_st, info = fbg.eval_population_accessibility(comps_st, G_base, dests,
                                                         avg_speed=60.0, # km/h
                                                         target_time_max = 0.25, # hours: it shouldn't take longer than this to reach any destination
                                                         target_pop_max = [0.80, 0.95], # fraction of population that should be reachable at each destination
                                                         length_attr = 'length_km',
                                                         population_attr = 'population',)

    min_comps_st = None
    return conn_pop_ratio, sys_st, min_comps_st

row_names = list(edges.keys()) 
n_state = 2 # binary states of components

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)


In [6]:
RSRPATH = Path("rsr_res") 

refs_mat_upper = torch.load(RSRPATH / f"refs_up_{sys_upper_st}.pt", map_location="cpu")
refs_mat_upper = refs_mat_upper.to(device)
refs_mat_lower = torch.load(RSRPATH / f"refs_low_{sys_upper_st-1}.pt", map_location="cpu")
refs_mat_lower = refs_mat_lower.to(device)

# Calculate system probabilities

## Marginal probability

In [9]:
pr_cond = rsr.get_comp_cond_sys_prob(
    refs_mat_upper,
    refs_mat_lower,
    probs,
    comps_st_cond = {},
    row_names = row_names,
    s_fun = s_fun,
    sys_upper_st = sys_upper_st
)
print(f"P(sys >= {sys_upper_st}) = {pr_cond['upper']:.3e}")
print(f"P(sys <= {sys_upper_st-1} ) = {pr_cond['lower']:.3e}\n")

P(sys >= 2) = 8.021e-01
P(sys <= 1 ) = 1.979e-01



## Conditional probability given "one" component's survival

In [ ]:
for x in row_names:
    print(f"Eval P(sys | {x}=1)")
    pr_cond = rsr.get_comp_cond_sys_prob(
        refs_mat_upper,
        refs_mat_lower,
        probs,
        comps_st_cond = {x: 1},
        row_names = row_names,
        s_fun = s_fun,
        sys_upper_st = sys_upper_st
    )
    print(f"P(sys >= {sys_upper_st} | {x}=1) = {pr_cond['upper']:.3e}")
    print(f"P(sys <= {sys_upper_st-1} | {x}=1) = {pr_cond['lower']:.3e}\n")

# Get multi-state probabilities

## First load refs for all states

In [12]:
RSRPATH = Path("rsr_res") 
refs_dict_mat_upper = {}
refs_dict_mat_lower = {}

for sys_upper_st in [1, 2]:  # either 1 or 2
    refs_mat_upper = torch.load(RSRPATH / f"refs_up_{sys_upper_st}.pt", map_location="cpu")
    refs_mat_upper = refs_mat_upper.to(device)
    refs_dict_mat_upper[sys_upper_st] = refs_mat_upper
    refs_mat_lower = torch.load(RSRPATH / f"refs_low_{sys_upper_st-1}.pt", map_location="cpu")
    refs_mat_lower = refs_mat_lower.to(device)
    refs_dict_mat_lower[sys_upper_st] = refs_mat_lower

## Marginal probability

In [13]:
# Initialize empty list to store the results
results = []

# Calculate probabilities
cond_probs = rsr.get_comp_cond_sys_prob_multi(
                refs_dict_mat_upper,
                refs_dict_mat_lower,
                probs,
                comps_st_cond = {},
                row_names = row_names,
                s_fun = s_fun
            )

# Print results
print("P(sys):", cond_probs)

# Append data as a dictionary to the list
results.append({"System failure": cond_probs[0],
                "Partial failure": cond_probs[1],
                "Survival": cond_probs[2]
                })

# Convert the list to a DataFrame
df_results = pd.DataFrame(results)

# Save to a JSON file
df_results.to_json("data/sys_probs.json", orient="records", indent=4)

print("\nData saved to data/sys_probs.json")

P(sys): {0: 0.000175, 1: 0.195388, 2: 0.804437}

Data saved to data/sys_probs.json


## Conditional probability given "one" component's survival

In [ ]:
# Initialize empty list to store the results
results = []

for x in row_names:
    # Calculate probabilities
    cond_probs = rsr.get_comp_cond_sys_prob_multi(
                    refs_dict_mat_upper,
                    refs_dict_mat_lower,
                    probs,
                    comps_st_cond = {x: 0}, # 1: survival, 0: failure
                    row_names = row_names,
                    s_fun = s_fun
                )

    # Print results
    print(f"P(sys | {x}=0):", cond_probs)

    # Append data as a dictionary to the list
    results.append({"Component": x,
                    "System failure": cond_probs[0],
                    "Partial failure": cond_probs[1],
                    "Survival": cond_probs[2]
                    })

# Convert the list to a DataFrame
df_results = pd.DataFrame(results)

# Save to a JSON file
df_results.to_json("data/cond_sys_probs.json", orient="records", indent=4)

print("\nData saved to data/cond_sys_probs.json")